In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
from joblib import Parallel, delayed
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ---- データの場所を自動で探す ----
def find(name):
    for dirname, _, filenames in os.walk("/kaggle/input"):
        if name in filenames:
            return os.path.join(dirname, name)
    raise FileNotFoundError(name)

train = pd.read_csv(find("train.csv")).rename(columns={"TA1.x": "TA1"})
test = pd.read_csv(find("test.csv"))

# ---- 入力: origin.ipynb と同じ8列 ----
features = [
    "Lat_Dec", "Lon_Dec", "PO4uM", "SiO3uM", "TA1",
    "Salinity1", "Temperature_degC", "R_Oxy_micromol.Kg",
]

# 標準化(統計量は訓練データのみから求める)
x_scaler = StandardScaler().fit(train[features])
X = x_scaler.transform(train[features])
X_test = x_scaler.transform(test[features])

y = train["DIC"].values
y_mu, y_sd = y.mean(), y.std()
y_scaled = (y - y_mu) / y_sd

# ---- 順伝播型ニューラルネットワーク(tanh, L-BFGS)のアンサンブル ----
# (隠れ層, alpha)
configs = [
    ((32, 16), 0.1),
    ((32,), 0.03),
    ((64, 32), 0.1),
    ((16, 8), 0.03),
]
N_MODELS = 36
SEED0 = 13000

def fit_predict(i):
    hidden, alpha = configs[i % len(configs)]
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="tanh",
        solver="lbfgs",
        alpha=alpha,
        max_iter=1000,
        random_state=SEED0 + i,
    )
    model.fit(X, y_scaled)
    return model.predict(X_test) * y_sd + y_mu

preds = Parallel(n_jobs=-1)(delayed(fit_predict)(i) for i in range(N_MODELS))
prediction = np.mean(preds, axis=0)

# ---- 提出ファイル ----
submission = pd.DataFrame({"id": test["id"], "DIC": prediction})
submission.to_csv("submission.csv", index=False)
print(submission.shape, submission["DIC"].isna().sum())
submission.head()

In [ ]:
# score: 3.85347
# name: 北野覚琉
# id: 62506239